# LedgerCall — Reliable & Efficient Finance Tool Calling on Tiny Aya
**Lexsi AI Safety & Alignment Hackathon | Track 3: Agentic Fine-Tuning**

End-to-end reproducible notebook on a single Kaggle/Colab **T4 (16 GB)** or **L4 (24 GB)** GPU:
1. Install **AgentTune** and **AuditKIT**.
2. Authenticate with Hugging Face (`CohereLabs/tiny-aya-global` is gated — accept its license on the model card first, then store `HF_TOKEN` in Kaggle Secrets or `os.environ`).
3. Generate the deterministic 600-train / 150-test LedgerCall benchmark with zero prompt leakage.
4. Evaluate the **base** `CohereLabs/tiny-aya-global` checkpoint (`--limit 100`) via AgentTune's rollout + AuditKIT's `agent_eval` scorers.
5. Fine-tune a **GRPO LoRA (`r=16`, `all-linear`)** policy using `create_agentic_trainer`.
6. Evaluate the **tuned** adapter (`--limit 100`) and render the before/after delta + Cost/ROI report.
7. Push the adapter, dataset, trajectories, and metrics to Hugging Face Hub and create public Model + Dataset collections.

In [ ]:
# 1. Install AgentTune, AuditKIT, and PEFT/Hub dependencies
!pip install -q "git+https://github.com/Lexsi-Labs/AgentTune" "auditkit[transformers]" peft datasets huggingface_hub


In [ ]:
# Ensure working directory is the LedgerCall repository root
import os
from pathlib import Path

REPO_URL = os.environ.get("LEDGERCALL_REPO_URL", "")
if not Path("ledgercall").is_dir() and Path("../ledgercall").is_dir():
    os.chdir("..")
elif not Path("ledgercall").is_dir() and REPO_URL:
    !git clone {REPO_URL} ledgercall_repo
    os.chdir("ledgercall_repo")

assert Path("ledgercall/tools.py").is_file(), "Run from the LedgerCall repository root (containing ./ledgercall)."
print("Working directory:", Path.cwd())


In [ ]:
# 2. Hugging Face authentication (Kaggle Secret `HF_TOKEN` or environment variable)
import os
from huggingface_hub import login

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        os.environ["HF_TOKEN"] = HF_TOKEN
    except Exception as exc:
        print(f"Kaggle secret HF_TOKEN not found ({exc}); set os.environ['HF_TOKEN'] before model download.")

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub.")

# Placeholder HF username — update before running Section 8 (Hub upload + collections)
HF_USER = os.environ.get("HF_USER", "TODO_YOUR_HF_USERNAME")
os.environ["HF_USER"] = HF_USER
print("HF_USER =", HF_USER)


In [ ]:
# 3. Generate deterministic train (600) and test (150) tasks + verify 0 prompt leakage
import json

!python -m ledgercall.data --out data/ --train-size 600 --test-size 150 --seed 42
print(json.dumps(json.loads(open("data/stats.json").read()), indent=2))


In [ ]:
# 4. Base evaluation (--limit 100) using AgentTune rollout + AuditKIT agent_eval
!python -m ledgercall.evaluate \
    --model CohereLabs/tiny-aya-global \
    --data data/test.jsonl \
    --tag base \
    --limit 100


In [ ]:
# 5. GRPO Agentic Fine-Tuning (LoRA r=16 all-linear, auto fp16 on T4 / bf16 on L4)
!python -m ledgercall.train \
    --model CohereLabs/tiny-aya-global \
    --train_file data/train.jsonl \
    --output_dir out/grpo \
    --max_steps 150 \
    --num_generations 4 \
    --per_device_train_batch_size 4 \
    --max_steps_per_turn 4


In [ ]:
# 6. Tuned adapter evaluation (--limit 100) on the identical test slice
!python -m ledgercall.evaluate \
    --model out/grpo \
    --data data/test.jsonl \
    --tag tuned \
    --limit 100


In [ ]:
# 7. Compare Before vs After metrics + Cost/ROI report
from IPython.display import Markdown, display

!python -m ledgercall.compare \
    --base results/base_metrics.json \
    --tuned results/tuned_metrics.json \
    --out results/comparison.md

display(Markdown(open("results/comparison.md").read()))


In [ ]:
# 8. Push LoRA adapter, datasets, trajectories & metrics to HF Hub + create public HF Collections
from huggingface_hub import add_collection_item, create_collection

if HF_USER and not HF_USER.startswith("TODO"):
    !HF_USER={HF_USER} python scripts/push_to_hub.py \
        --adapter-dir out/grpo \
        --data-dir data \
        --results-dir results

    model_repo = f"{HF_USER}/ledgercall-tiny-aya-grpo"
    dataset_repo = f"{HF_USER}/ledgercall-finance-toolbench"

    model_col = create_collection(
        title="LedgerCall Models (Lexsi Hackathon Track 3)",
        namespace=HF_USER,
        description="GRPO LoRA adapter teaching CohereLabs/tiny-aya-global deterministic finance tool calling.",
        private=False,
        exists_ok=True,
    )
    add_collection_item(model_col.slug, item_id=model_repo, item_type="model", exists_ok=True)

    data_col = create_collection(
        title="LedgerCall Datasets & Trajectories (Lexsi Hackathon Track 3)",
        namespace=HF_USER,
        description="Synthetic finance tool-calling splits, AgentTune trajectories, and AuditKIT evaluation reports.",
        private=False,
        exists_ok=True,
    )
    add_collection_item(data_col.slug, item_id=dataset_repo, item_type="dataset", exists_ok=True)

    print("Model Collection URL:  ", model_col.url)
    print("Dataset Collection URL:", data_col.url)
else:
    print("Set HF_USER (not starting with 'TODO') in Cell 3 to push artifacts and create Hub collections.")
